# Homework — Part B: Fixing a Model's Thinking Loops by Editing Its Weights

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/homework/weight-space/HW-Part-B-Fixing-Loops.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Recommended environment: Google Colab with a T4 GPU (`Runtime → Change runtime type → T4 GPU`).

In [ ]:
!pip install -q -U flash-linear-attention
# Download the reference solutions for this notebook's exercises (q_*.py)
import os, urllib.request
BASE_URL = 'https://raw.githubusercontent.com/tjwei/DataScienceSoftware2026/main/homework/weight-space/'
for fn in ['hwtools.py']:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(BASE_URL + fn, fn)

## Background

With thinking turned on, the small model **Qwen3.5-0.8B** often gets stuck: it starts doubting itself ("Wait, if…"), and eventually repeats the same sentences until it runs out of tokens. On MATH-500, 93% of its answers hit the 32,768-token limit, so it scores only 7.9%.

In [*Sparks of Weight-Space Programming*](https://rikiyat.github.io/blog/weight-space-programming/) (experiment 8), the LLM "GPT-6 Astra" fixed this **without any training**: it changed 64 columns of `mlp.down_proj` in layers 15 and 19 (0.016% of the weights), and MATH-500 went from 7.9% to 48.1%. A follow-up by [nathan.rs](https://nathan.rs/posts/weight-space-lp/) reproduced the fix with standard tools.

**Your job: find and fix the loop yourself, with known methods**: difference of means, rank-one edits, controls, and careful evaluation. You may use an LLM to help, but every step must be backed by numbers or figures you produced. The steps below are guided: each 📌 asks you to *observe* something and explain it.

Runtime: Colab **T4**. Reading traces and teacher-forced evaluation take minutes; one generation evaluation (64 problems, 4096 tokens) takes about 15 minutes.

In [ ]:
import time, json
import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM
import hwtools as H

tok = AutoTokenizer.from_pretrained(H.MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(H.MODEL_ID, dtype=torch.float16, device_map="cuda").eval()
layers = model.model.layers
print(len(layers), "layers; hidden size", model.config.hidden_size, "; MLP width", model.config.intermediate_size)

## The data

The authors released the unedited model's answers to 200 GSM8K and 200 MATH-500 problems (thinking on, up to 32k tokens each). We truncate them to the first 8192 tokens, which still contains the start of most loops.

In [ ]:
traces = H.load_traces(max_tokens=8192, tokenizer=tok)
onsets = [H.loop_onset(t["ids"]) for t in traces]

# Hold out 40 traces (20 GSM8K + 20 MATH-500), fixed by problem id, for checking edits in B4.
# Directions and columns are estimated only from the other traces (dev_traces).
probe_ids = {t["id"] for t in traces[1::10]}
probe_traces = [t for t in traces if t["id"] in probe_ids]
dev_traces = [t for t in traces if t["id"] not in probe_ids]
assert probe_ids.isdisjoint(t["id"] for t in dev_traces)
print(len(dev_traces), "traces for estimating directions,", len(probe_traces), "held out for B4")
print(len(traces), "traces;", sum(o is not None for o in onsets), "contain a loop within 8192 tokens")
plt.hist([o for o in onsets if o is not None], bins=30); plt.xlabel("token where the first loop starts"); plt.ylabel("traces"); plt.show()

## B1. What does a loop look like?

`H.loop_onset` uses a simple rule: a token is *repeated* if the 16 tokens ending at it have appeared before; a loop starts where at least 64 repeated tokens follow each other.

In [ ]:
t = next(t for t, o in zip(traces, onsets) if o is not None)
o = H.loop_onset(t["ids"])
print(">>> 400 tokens before the loop:\n", tok.decode(t["ids"][max(o - 400, 0):o]))
print("\n>>> the loop:\n", tok.decode(t["ids"][o:o + 150]))

📌 **B1.** Read at least five traces. Does a loop start suddenly, or is there a phase before it? Describe that phase. Does the 16-gram rule detect it? What does that mean for the tokens labelled "productive" by `H.label_tokens` (see its docstring)?

## B2. Reverse-engineer Astra's edit

Astra's edited values are public. Load them and compute the change `ΔW` for layers 15 and 19.

In [ ]:
astra = H.load_astra_edit(model)
astra_dirs = {}
for L, Wnew in astra.items():
    D = (Wnew.float() - H.down_proj(model, L).float())          # (1024, 3584)
    cols = D.abs().sum(0).nonzero().squeeze(1)
    print(f"layer {L}: {len(cols)} columns changed")
    # TODO: is D[:, cols] rank one? (look at its singular values)
    # TODO: if so, take u = the first left singular vector, and find alpha from  D[:, j] = -alpha (u . w_j) u
    # astra_dirs[L] = (u, cols)

📌 **B2.** Is the edit rank one? What is `alpha`? Project `u` (and `-u`) through the final norm and `lm_head` (a logit lens): which tokens does each end of `u` promote? Does that match the B1 observations?

## B3. Find a "loop direction" yourself

### B3a. Difference of means

Collect the residual stream right after layers 15 and 19 (`hidden_states[16]`, `hidden_states[20]`), and the MLP neuron activations (the input of `down_proj`), on looping and on productive tokens.

In [ ]:
acts = {}
hooks = [layers[L].mlp.down_proj.register_forward_pre_hook(lambda m, inp, L=L: acts.__setitem__(L, inp[0][0])) for L in (15, 19)]
sums = {k: torch.zeros(len(layers) + 1, 1024, device="cuda", dtype=torch.float64) for k in (0, 1)}
act_sums = {(L, k): torch.zeros(3584, device="cuda", dtype=torch.float64) for L in (15, 19) for k in (0, 1)}
counts = {0: 0, 1: 0}
t0 = time.time()
dom_traces = dev_traces[::3]                             # a third of the dev traces is enough (~5 minutes)
for t in dom_traces:
    lab = H.label_tokens(t["ids"])
    p = tok(H.prompt(t["problem"]), add_special_tokens=False).input_ids
    ids = torch.tensor([p + t["ids"]], device="cuda")
    with torch.no_grad():
        out = model(ids, output_hidden_states=True, logits_to_keep=1)
        hs = torch.stack(out.hidden_states)[:, 0, len(p):]
        for k in (0, 1):
            m = torch.tensor(lab == k, device="cuda")
            if m.any():
                sums[k] += hs[:, m].double().sum(1); counts[k] += int(m.sum())
                for L in (15, 19): act_sums[(L, k)] += acts[L][len(p):][m].double().sum(0)
    del out, hs
for h in hooks: h.remove()
mean_diff = (sums[1] / counts[1] - sums[0] / counts[0]).float()
print(counts, f"{time.time()-t0:.0f}s")
plt.plot(mean_diff.norm(dim=1).cpu()); plt.xlabel("layer"); plt.ylabel("|mean(loop) - mean(productive)|"); plt.show()

In [ ]:
def top_columns(L, u, act_diff, k=64):
    """columns of down_proj that write the most along u, weighted by how much more they fire on looping tokens"""
    W = H.down_proj(model, L).float()
    return ((u @ W) * act_diff).argsort(descending=True)[:k]

dom = {}
for L in (15, 19):
    u = mean_diff[L + 1] / mean_diff[L + 1].norm()
    act_diff = (act_sums[(L, 1)] / counts[1] - act_sums[(L, 0)] / counts[0]).float()
    dom[L] = (u, top_columns(L, u, act_diff))
    # TODO: compare with Astra: cosine between u and Astra's u, and how many of the 64 columns are the same

### B3b. Matched pairs

The difference of means compares tokens from *different* places: looping tokens come late in long answers, productive tokens mostly early. Astra instead wrote **matched pairs**: the same problem and the same reasoning up to a point, then one *productive* continuation and one *repetitive* one. For example (one of Astra's four pairs):

In [ ]:
PAIR_EXAMPLE = (
    "<|im_start|>user\nA machine makes 45 necklaces, and another makes 2.4 times as many. Find the total.<|im_end|>\n<|im_start|>assistant\n<think>\n"
    "The second machine makes 45*2.4=108 necklaces. Add the first machine's 45 to obtain 153 necklaces. "
    "Multiplication gives the second quantity, while addition combines the two machines. A check is 45*3.4=153.",
    "<|im_start|>user\nA machine makes 45 necklaces, and another makes 2.4 times as many. Find the total.<|im_end|>\n<|im_start|>assistant\n<think>\n"
    "The second machine makes 45*2.4=108 necklaces. Add the first machine's 45 to obtain 153 necklaces. "
    "Is 108 the total? No, 45+108=153. Is 108 the total? No, 45+108=153. Is 108 the total? No, 45+108=153.",
)

def branch_mean(text, n_prefix):
    """mean residual (after layers 15 and 19) and neuron activations over the tokens after the shared prefix"""
    ids = tok(text, add_special_tokens=False).input_ids
    hooks = [layers[L].mlp.down_proj.register_forward_pre_hook(lambda m, inp, L=L: acts.__setitem__(L, inp[0][0])) for L in (15, 19)]
    with torch.no_grad():
        out = model(torch.tensor([ids], device="cuda"), output_hidden_states=True, logits_to_keep=1)
    for h in hooks: h.remove()
    return ({L: out.hidden_states[L + 1][0, n_prefix:].float().mean(0) for L in (15, 19)},
            {L: acts[L][n_prefix:].float().mean(0) for L in (15, 19)})

# TODO: write at least FOUR pairs of your own (different problems; vary the kind of repetition), e.g. with an LLM's help.
MY_PAIRS = [PAIR_EXAMPLE]

In [ ]:
pairs = {}
res, act = {L: 0 for L in (15, 19)}, {L: 0 for L in (15, 19)}
for productive, repetitive in MY_PAIRS:
    a, b = (tok(x, add_special_tokens=False).input_ids for x in (productive, repetitive))
    n = next(i for i in range(min(len(a), len(b))) if a[i] != b[i])          # shared prefix length
    (rp, ap), (rr, ar) = branch_mean(productive, n), branch_mean(repetitive, n)
    for L in (15, 19):
        res[L] = res[L] + rr[L] - rp[L]; act[L] = act[L] + ar[L] - ap[L]
for L in (15, 19):
    u = res[L] / res[L].norm()
    pairs[L] = (u, top_columns(L, u, act[L]))
    # TODO: cosine with Astra's u and with the difference-of-means u; column overlap

📌 **B3.** Compare the two directions with Astra's (cosine, column overlap). Also compute a **position direction** (mean of productive tokens late in the answer minus early in the answer) and its cosine with each `u`. Which direction picks up more of things you did not intend to measure, and why? How does the result change with the number and the variety of your pairs?

## B4. Dose–response with a cheap metric

`H.teacher_forced` feeds the saved traces to the (edited) model and measures the mean log-probability of the actual tokens on three groups: **looping** tokens, the 64 tokens where each **loop starts**, and **productive** tokens. A good edit should make looping less likely without making productive text less likely.

We measure on `probe_traces`, the 40 traces held out at the start. An edit is estimated to push looping tokens down, so measuring it on the very traces it was estimated from would flatter it. Keep any trace you use in B3 (including for the position direction) out of this set.

In [ ]:
assert probe_ids.isdisjoint(t["id"] for t in dom_traces)   # B4 must not reuse the traces the direction was estimated from
base_tf = H.teacher_forced(model, tok, probe_traces)
print("unedited", base_tf)

In [ ]:
# example: one point of the dose-response curve
restore = H.rank_one_edit(model, dom, alpha=6.0)
print("difference of means, alpha 6", H.teacher_forced(model, tok, probe_traces))
restore()
# TODO: sweep alpha in (1, 2, 4, 6, 12) and the number of columns k in (8, 64, all 3584), for your directions and for Astra's

📌 **B4.** Plot the curves. Which group of tokens reacts most to the edit? Keep these numbers: in B6 you will check whether the cheap metric predicts what really matters.

## B5. Controls

* a **random direction**, with the same columns and the same norm of change;
* the edit on **only one** of the two layers, or on **other** layers;
* (optional) a single neuron instead of 64.

In [ ]:
# TODO: the controls (teacher-forced metric first; generation for the most informative ones)

📌 **B5.** Which controls have no effect? What does each control rule out?

## B6. Generation, and the evaluation budget

The real goal is to stop the loops **when the model generates**. `H.generate_eval` generates answers to 64 GSM8K problems that are not among the traces, with the paper's sampling settings, and reports the loop rate, the share that finished thinking (`</think>`), and the accuracy.

In [ ]:
problems = H.eval_problems(64)
summary, rows = H.generate_eval(model, tok, problems, max_new_tokens=4096)
print("unedited @4096:", summary)

In [ ]:
astra_restore = {}
for L, Wnew in astra.items():
    astra_restore[L] = H.down_proj(model, L).data.clone(); H.down_proj(model, L).data.copy_(Wnew)
summary, rows = H.generate_eval(model, tok, problems, max_new_tokens=4096)
print("Astra @4096:", summary)
for L, w in astra_restore.items(): H.down_proj(model, L).data.copy_(w)
# TODO: your edits; at least two seeds for the important ones (seed=1, seed=2)

📌 **B6a.** Does the teacher-forced metric of B4 predict the generation results? Which of its three numbers is most useful, and which is misleading?

📌 **B6b.** The accuracy at 4096 tokens is low even for Astra's edit. Look at the lengths of the edited model's *correct* answers (`rows`): why can a 4096-token budget measure the loop rate but not the accuracy? Run one comparison at a longer budget on a few problems, e.g. `H.generate_eval(model, tok, problems[:16], max_new_tokens=16384)` (about 45 minutes per model on a T4), and look at how the share that finished thinking grows with the budget.

📌 **B6c.** How large is the difference between two seeds of the same model? Which of your differences are larger than that?

## B7. Side effects

* With **thinking off** (`thinking=False`), the unedited model is much better (it does not loop). Does your edit hurt it?
* The productive tokens in B4.

In [ ]:
# TODO: thinking-off accuracy (short answers, fast) with and without your edit

📌 **B7.** What did the edit cost? Is there a strength `alpha` where the loops stop but the model also stops working?

## B8. Close the gap (open-ended)

Try to match or beat Astra's edit, and explain where the remaining difference comes from. Ideas: better pairs; contrasting the start of loops instead of their middle; choosing columns by solving a small optimisation problem (nathan.rs used a linear program with an L1 objective, which picks only 8 columns per layer); replacing the weight edit by a constant steering vector.

## Report

Write up B1–B8 as a chain of evidence: what you observed, what each observation rules out, and which of your conclusions still lack evidence.

## References

* Takehi, Teoh, Zhang, Khattab: [Sparks of Weight-Space Programming](https://rikiyat.github.io/blog/weight-space-programming/) (2026); logs and data: [RikiyaT/lm_understand-logs](https://github.com/RikiyaT/lm_understand-logs) (CC BY 4.0)
* [Weight-space programming with linear programming](https://nathan.rs/posts/weight-space-lp/) (nathan.rs)
* [What Qwen is thinking when it gets stuck saying "Wait"](https://alisorcorp.github.io/inside-a-reasoning-loop/)
* [Qwen3.5-0.8B-Astra-WeightEdit](https://huggingface.co/JonathanColetti/Qwen3.5-0.8B-Astra-WeightEdit): the edited model and an independent benchmark